In [2]:
import pandas as pd
from typing import List,Dict,Any
import os

In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter   , 
    CharacterTextSplitter ,
    TokenTextSplitter
)
print("setup completed")

d:\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup completed


Understanding documents structure


In [4]:
## create a single document
doc = Document(
    page_content= "This is a test document. It contains some text that we will use to test the text splitters.",
    metadata={"source": "example.txt",
              "page": 1,
              "author": "John Doe",
              "date_created": "2023-01-01",
              "custom_field": "any_value"
              }
)
print("Document structure created:")
print(f"content :{doc.page_content}")
print(f"metadata :{doc.metadata}")

Document structure created:
content :This is a test document. It contains some text that we will use to test the text splitters.
metadata :{'source': 'example.txt', 'page': 1, 'author': 'John Doe', 'date_created': '2023-01-01', 'custom_field': 'any_value'}


In [5]:
type(doc)

langchain_core.documents.base.Document

In [6]:
import os
os.makedirs("data/text_files", exist_ok=True)

In [8]:
sample_text = {
    "data/text_files/waterscarcity.txt":"""Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is broadly categorized into two types:

Physical Water Scarcity: Occurs when natural water resources are insufficient to meet a region's demand, often in arid or semi-arid regions.

Economic Water Scarcity: Occurs when poor management, lack of infrastructure, or under-investment prevents communities from accessing available water.

Key Causes
Climate Change: Alters weather patterns, leading to prolonged droughts, unpredictable rainfall, and shrinking glaciers.

Over-extraction: Excessive pumping of groundwater for agriculture, industry, and domestic use faster than it can naturally replenish.

Pollution: Contamination of freshwater sources from industrial waste, agricultural run-off, and inadequate sewage treatment, making water unsafe for use.

Population Growth: Rising global demand combined with urbanization places immense strain on local water supplies.

Major Impacts
Food Insecurity: Agriculture consumes roughly 70% of global freshwater; scarcity directly threatens crop yields and livestock.

Health Crises: Lack of clean drinking water leads to sanitation issues and the spread of waterborne diseases like cholera and dysentery.

Economic & Social Disruption: Water shortages displace populations, hinder industrial output, and can trigger regional conflicts over shared river basins.

Sustainable management requires improving irrigation efficiency, investing in wastewater recycling, protecting watersheds, and deploying desalination technologies where appropriate.""",

"data/text_files/deforestation.txt":"""Understanding Deforestation
Deforestation is the large-scale clearing, thinning, or removal of forests, typically resulting from human activities like agriculture, logging, and urbanization. While forests once covered nearly half of all land on Earth, they now account for less than one-third, with millions of acres lost every year.

Key Drivers of Deforestation
Agricultural Expansion: The conversion of forest land into crop fields and livestock pastures is responsible for the vast majority of global deforestation.

Commercial Logging: The demand for timber, paper, and wood products drives both legal and illegal tree harvesting, often opening up pristine forests to further destruction.

Infrastructure Development: Building roads, highways, dams, and urban areas fragments forest ecosystems and accelerates human encroachment.

Mining and Extraction: Resource extraction clears large swaths of land and frequently introduces toxic pollutants into surrounding ecosystems.

Major Consequences
Biodiversity Loss: Forests house the majority of Earth's terrestrial plants and animals. Their destruction leads to habitat fragmentation, pushing countless species toward extinction.

Climate Change: Trees absorb and store carbon dioxide. When they are cut down or burned, that stored carbon is released back into the atmosphere, intensifying global warming.

Disruption of the Water Cycle: Forests regulate precipitation and soil moisture. Deforestation can lead to severe droughts, altered weather patterns, and degraded soil quality prone to erosion.

Impact on Indigenous Communities: Millions of people rely directly on forests for their livelihoods, food, and cultural survival, making them uniquely vulnerable to land clearing.

Solutions and Mitigation
Stopping deforestation requires a multi-pronged approach combining global policy, sustainable corporate practices, and local conservation efforts.

Strengthening Land Rights: Empowering indigenous and local communities to legally protect their ancestral forests.

Sustainable Supply Chains: Enforcing strict zero-deforestation commitments from major corporations regarding commodities like palm oil, beef, and soy.

Reforestation and Afforestation: Actively planting trees in degraded landscapes and restoring natural ecosystems to rebuild carbon sinks and habitats.

Eco-Friendly Consumer Choices: Supporting certified sustainable products (such as FSC-certified wood and paper) to reduce market demand for destructive practices."""
}

for file_path,content in sample_text.items():
    with open(file_path,'w', encoding="utf-8") as f:
        f.write(content)
print ("samples created")        


samples created


Text Loader -single file Reading


In [11]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("data/text_files/waterscarcity.txt", encoding="utf-8")
documents=loader.load()
print(f"loaded {len(documents)} documents")
print(f"content preview: {documents[0].page_content[:100]}...")
print(f"metadata:{documents[0].metadata}")



loaded 1 documents
content preview: Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is ...
metadata:{'source': 'data/text_files/waterscarcity.txt'}


In [14]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
documnents=dir_loader.load()
print(f"loaded {len(documnents)} documents")
for i , doc in enumerate(documnents):
    print(f"Document {i+1}:")
    print(f"source: {doc.metadata['source']}")
    print(f"Length: {len(doc.page_content)} chacters")
    


100%|██████████| 2/2 [00:00<00:00, 1265.63it/s]

loaded 2 documents
Document 1:
source: data\text_files\deforestation.txt
Length: 2491 chacters
Document 2:
source: data\text_files\waterscarcity.txt
Length: 1585 chacters


In [16]:
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter   , 
    CharacterTextSplitter ,
    TokenTextSplitter
)

In [17]:
# Character text splitter
text=documents[0].page_content
text

"Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is broadly categorized into two types:\n\nPhysical Water Scarcity: Occurs when natural water resources are insufficient to meet a region's demand, often in arid or semi-arid regions.\n\nEconomic Water Scarcity: Occurs when poor management, lack of infrastructure, or under-investment prevents communities from accessing available water.\n\nKey Causes\nClimate Change: Alters weather patterns, leading to prolonged droughts, unpredictable rainfall, and shrinking glaciers.\n\nOver-extraction: Excessive pumping of groundwater for agriculture, industry, and domestic use faster than it can naturally replenish.\n\nPollution: Contamination of freshwater sources from industrial waste, agricultural run-off, and inadequate sewage treatment, making water unsafe for use.\n\nPopulation Growth: Rising global demand combined with urbanization places immense strain on local water supplies.\n\nMajor Impacts\nF

In [18]:
#character based text splitter
print ("CharacterTextSplitter")
character_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
char_chunks = character_splitter.split_text(text)
print(f"Number of chunks: {len(char_chunks)}")
print(f"First chunk preview: {char_chunks[0][:100]}...")   

CharacterTextSplitter
Number of chunks: 11
First chunk preview: Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is ...


In [21]:
print(len(char_chunks[0]))
print (char_chunks[0])

print("-------------")
print(len(char_chunks[1]))

print (char_chunks[1])
print("-------------")
print(len(char_chunks[2]))

print (char_chunks[2])
print("-------------")

print (char_chunks[3])

135
Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is broadly categorized into two types:
-------------
140
Physical Water Scarcity: Occurs when natural water resources are insufficient to meet a region's demand, often in arid or semi-arid regions.
-------------
161
Economic Water Scarcity: Occurs when poor management, lack of infrastructure, or under-investment prevents communities from accessing available water.
Key Causes
-------------
Key Causes
Climate Change: Alters weather patterns, leading to prolonged droughts, unpredictable rainfall, and shrinking glaciers.


In [ ]:
#method 2: RecursiveCharacterTextSplitter
print ("RecursiveCharacterTextSplitter")
reccursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n", "\n\n","", " "],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
recursive_chunks = reccursive_splitter.split_text(text)
print(f"Number of chunks: {len(recursive_chunks)}")
print(f"First chunk preview: {recursive_chunks[0][:100]}...")



RecursiveCharacterTextSplitter
Number of chunks: 11
First chunk preview: Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is ...


In [25]:
print(len(recursive_chunks[0]))
print(recursive_chunks[0])
print("-------------")
print(len(recursive_chunks[1]))
print(recursive_chunks[1])
print("-------------")
print(len(recursive_chunks[2]))
print(recursive_chunks[2])
print("-------------")
print(len(recursive_chunks[3]))
print(recursive_chunks[3])

135
Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is broadly categorized into two types:
-------------
140
Physical Water Scarcity: Occurs when natural water resources are insufficient to meet a region's demand, often in arid or semi-arid regions.
-------------
162
Economic Water Scarcity: Occurs when poor management, lack of infrastructure, or under-investment prevents communities from accessing available water.

Key Causes
-------------
130
Key Causes
Climate Change: Alters weather patterns, leading to prolonged droughts, unpredictable rainfall, and shrinking glaciers.


In [34]:
simple_text="""Water scarcity refers to the lack of fresh water resources to meet the standard water demand. It is broadly categorized into two types"""
splitter=RecursiveCharacterTextSplitter(
    separators=[" "],
    chunk_size=40,
    chunk_overlap=10,
    length_function=len

)
chunks = splitter.split_text(simple_text)
print(f"Number of chunks: {len(chunks)}")

for i in range(len(chunks)):
    print(f"Chunk {i+1}: {chunks[i]} (Length: {len(chunks[i])})")

    print()

Number of chunks: 5
Chunk 1: Water scarcity refers to the lack of (Length: 36)

Chunk 2: lack of fresh water resources to meet (Length: 37)

Chunk 3: to meet the standard water demand. It (Length: 37)

Chunk 4: It is broadly categorized into two (Length: 34)

Chunk 5: into two types (Length: 14)

